In [1]:
import Pkg
Pkg.activate("../../.")

  Activating project at `~/dev/MyCloudAtlas.jl`


# Minimal findsoln debug (Re=300)

This notebook loads a single candidate from `solutions.bin` and runs one
Channelflow `findsoln` in an isolated output directory.

In [2]:
using CloudAtlas
using Serialization
using Dates
using ChannelflowWrapper
using Base.Threads

## Configuration

In [3]:
# Domain sizes (α = 2π/Lx, γ = 2π/Lz)
α, γ = 2π/6.0, 2π/4.0
Re = 300.0

# Pick the symmetry group and discretization you want to test
symm_name = "sxytxz"
J, K, L = 1, 2, 3

# Which solutions in solutions.bin?
solution_idx = 1
solution_idx2 = 2

# Paths
base_dir = @__DIR__
solutions_path = joinpath(base_dir, "tw_discovery_re300", symm_name, "jkl_$(J)_$(K)_$(L)", "solutions.bin")
reference_path = joinpath(base_dir, "TW1-2pi1piRe200-40x49x40.nc")
reference_field_converted = joinpath(base_dir, "tw_discovery_re300", "reference_field_$(α)_$(γ).nc")

symm_file = joinpath(base_dir, "$(symm_name).asc")

# findsoln settings
T = 10.0

10.0

## Load one candidate

In [4]:
solutions = open(solutions_path, "r") do io
    deserialize(io)
end

@assert 1 <= solution_idx <= length(solutions)
@assert 1 <= solution_idx2 <= length(solutions)

## Build model + run two findsoln jobs

In [5]:
sx, sy, sz, tx, tz = CloudAtlas.halfbox_symmetries()
H = [(sx * sy) * (tx * tz)]

model = ODEModel(α, γ, J, K, L, H;
    normalize = false,
    tw = true,
)

# Ensure reference field exists at the requested α,γ
if !isfile(reference_field_converted)
    changegrid(reference_path, reference_field_converted; al = α, ga = γ)
end

function run_findsoln(idx::Int)
    ξ = solutions[idx]
    x, cx, cz = extract_components(ξ, model)

    stamp = Dates.format(now(), "MM-DD-HHMMSS")
    sol_dir = joinpath(base_dir, "tw_discovery_re300", "debug_findsoln", "sol_$(idx)_$(stamp)")
    mkpath(sol_dir)

    guess_path = joinpath(sol_dir, "u_guess.nc")
    sigma_file = joinpath(sol_dir, "sigma.asc")

    println("[Thread $(threadid())] coeff2field idx=$(idx)")
    coeff2field(x, model.ijkl, reference_field_converted, guess_path; workdir = sol_dir)
    println("[Thread $(threadid())] save_sigma idx=$(idx)")
    save_sigma(model, cx, cz, T, sigma_file)

    println("[Thread $(threadid())] findsoln start idx=$(idx)")
    findsoln(guess_path;
        workdir = sol_dir,
        R = Re,
        eqb = true,
        xrel = model.keep_cx,
        zrel = model.keep_cz,
        symms = abspath(symm_file),
        sigma = sigma_file,
        od = sol_dir,
        T = T,
    )
    println("[Thread $(threadid())] findsoln done idx=$(idx)")
end

indices = [solution_idx, solution_idx2]
Threads.@threads for i in 1:length(indices)
    run_findsoln(indices[i])
end

J,K,L,m == 1,2,3,53
(2J+1)(2K+1)(2L+1) + 1 == 106
Making matrices B,A1,A2,S3...
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 
Making matrices Cx,Cz...
Phase constraints: keep_cx = false, keep_cz = true
[Thread 4] coeff2field idx=2
[Thread 2] coeff2field idx=1
alpha, gamma == 1.047197551196598, 1.570796326794897
Nx, Ny, Nz == 40, 49, 40
Reading ijkl indices of basis set from file
reading N == 53 ijkl indices
ijkl[0] == 1 0 0 1
L == max l == 3
Constructing Legendre polynomials
Assigning Polynomial, size = 1 
Assigning Polynomial, size = 0 1 
Assigning Polynomial, size = -0.5 0 1.5 
Assigning Polynomial, size = 0 -1.5 0 2.5 
Constructing S, and Sprime polynomials
Assigning Polynomial, size = 0 1 0 -0.3333333333333333 
Assigning Polynomial, size = 1 0 -2 0 1 
Assigning Polynomial, size = 0 1 0 -2 0 1 
Assigning Polynomial, size = -0.5 0 2.5 0 -3.5 0 1.5 
A

LoadError: InterruptException: